# 01. OpenAI API 기초
**SK하이닉스 Autonomous R&D — Day 3** 

> ChatGPT 웹과 **같은 모델**을 코드에서 호출.

---
## 0. 라이브러리 설치

아래 셀을 **최초 1회** 실행.

In [1]:
!pip install openai==1.58.1

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 20.2 MB/s  0:00:00
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [openai]14/15 [openai]c]core]


In [2]:
!pip install openai python-dotenv

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)


---
## 1. LLM API란?

강의자료 요약:

| 구분 | ChatGPT 웹 | API |
|------|------------|-----|
| 사용자 | 사람이 직접 입력 | **프로그램**이 요청 |
| 결과 | 화면에 표시 | **response 객체**로 반환 |
| 본질 | 대화 | **Prompt → Text** |

API 요청의 핵심 3요소:
1. **`model`** — 어떤 LLM을 쓸지
2. **`messages`** — system / user (대화 내용)
3. **`temperature`** — 답변의 무작위성 (0=일관, 1=창의)

---
## 2. API 키 연결


처음에는 **키를 변수에 넣어** 바로 연결해 봅니다.

> OpenAI Platform → [API keys](https://platform.openai.com/api-keys) 에서 발급

In [9]:
from pathlib import Path

print((Path.cwd().parent / ".env").exists())
print(Path.cwd().parent / ".env")

False
/Users/seorincho/Desktop/code/2026_AI/SK Autonomous R&D/.env


In [10]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

api_key = os.getenv("OPENAI_KEY")

print(api_key)

REDACTEDuDBKlCP3sc-TBybmvQQac7XGFvOANuNCa9GcbMykmUAq8XP3JLxm4l638KBywPU4lLL4Dm_274T3BlbkFJQhOfJH1taggwOaixxYjnOEsuna3qCloEGOVxLtrHlzq0IhmQa-FcIx7PBRmcDttf4Jdn7EhNYA


In [11]:
# 연결 테스트
test = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': 'Hi, reply with one word: OK'}],
    max_tokens=10,
)
print('연결 성공:', test.choices[0].message.content)

연결 성공: OK


1. **`.env` 파일**에 키 저장 (코드와 분리) (OPENAI_API_KEY=sk...)
2. **`.gitignore`**에 `.env` 등록 → Git에 **절대 올리지 않음**

In [13]:
from pathlib import Path

root = Path('..')  # ttest/
env_path = root / '../.env'
gitignore_path = root / '../.gitignore'

print('.env 존재:', env_path.exists(), '→', env_path.resolve())
print('.gitignore 존재:', gitignore_path.exists())

if gitignore_path.exists():
    content = gitignore_path.read_text(encoding='utf-8')
    print('.env in .gitignore:', '.env' in content)

.env 존재: True → /Users/seorincho/Desktop/code/2026_AI/.env
.gitignore 존재: True
.env in .gitignore: True


In [18]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")
api_key = os.getenv('OPENAI_KEY')

if not api_key:
    raise ValueError('ttest/.env에 OPENAI_API_KEY=sk-... 를 설정하세요.')

client = OpenAI(api_key=api_key)
print('✅ .env에서 API 키 로드 완료 (코드에 키 없음)')

✅ .env에서 API 키 로드 완료 (코드에 키 없음)


---
## 3. 첫 API 호출 

`chat.completions.create()` — **messages** 리스트를 보내면 AI가 답변합니다.

| role | 역할 |
|------|------|
| `system` | AI 역할·규칙 (선택) |
| `user` | 사용자 질문 |

In [19]:
response = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.1,
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user',   'content': '2002년 월드컵 우승 팀이 어디야?'},
    ],
)

In [24]:

response.choices[0].message.content

'2002년 월드컵 우승 팀은 브라질입니다. 브라질은 결승에서 독일을 2-0으로 이기고 5번째 월드컵 트로피를 차지했습니다.'

In [26]:
####### 2022년 월드컵 우승팀 물어보기
response = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.1,
    messages=[
        {'role': 'system', 'content': 'You are a professional sports analyst.'},
        {'role': 'user',   'content': '2026년 월드컵 우승 확률이 제일 높은 나라가 어디야?'},
    ],
)


In [27]:

response.choices[0].message.content

'2026년 월드컵 우승 확률이 가장 높은 나라는 여러 요인에 따라 달라질 수 있지만, 일반적으로 축구 강국으로 알려진 나라들이 우승 후보로 꼽힙니다. 예를 들어, 브라질, 독일, 프랑스, 아르헨티나, 이탈리아, 스페인 등은 역사적으로 월드컵에서 성공을 거둔 팀들입니다.\n\n특히 프랑스와 아르헨티나는 최근 대회에서 좋은 성적을 거두었고, 브라질은 항상 강력한 팀으로 평가받고 있습니다. 또한, 독일과 이탈리아는 전통적으로 강한 팀으로, 언제든지 우승 후보로 떠오를 수 있습니다.\n\n최신 데이터와 팀의 현재 상태, 선수들의 폼, 부상 여부 등을 고려해야 정확한 예측이 가능하지만, 일반적으로 위의 나라들이 우승 확률이 높은 팀으로 예상됩니다. 2026년 월드컵이 북미에서 개최되는 만큼, 홈 어드밴티지를 가진 미국과 캐나다도 주목할 만한 팀이 될 수 있습니다.'

In [28]:
response

ChatCompletion(id='chatcmpl-DuBDlvkiN6USdJqYeT370wBffZh9N', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='2026년 월드컵 우승 확률이 가장 높은 나라는 여러 요인에 따라 달라질 수 있지만, 일반적으로 축구 강국으로 알려진 나라들이 우승 후보로 꼽힙니다. 예를 들어, 브라질, 독일, 프랑스, 아르헨티나, 이탈리아, 스페인 등은 역사적으로 월드컵에서 성공을 거둔 팀들입니다.\n\n특히 프랑스와 아르헨티나는 최근 대회에서 좋은 성적을 거두었고, 브라질은 항상 강력한 팀으로 평가받고 있습니다. 또한, 독일과 이탈리아는 전통적으로 강한 팀으로, 언제든지 우승 후보로 떠오를 수 있습니다.\n\n최신 데이터와 팀의 현재 상태, 선수들의 폼, 부상 여부 등을 고려해야 정확한 예측이 가능하지만, 일반적으로 위의 나라들이 우승 확률이 높은 팀으로 예상됩니다. 2026년 월드컵이 북미에서 개최되는 만큼, 홈 어드밴티지를 가진 미국과 캐나다도 주목할 만한 팀이 될 수 있습니다.', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None, annotations=[]))], created=1782281873, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_8e7ffbf8f8', usage=CompletionUsage(completion_tokens=247, prompt_tokens=37, total_tokens=284, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_toke

In [29]:
response.usage.total_tokens

284

---
## 4. 응답(Response) 객체

API는 **텍스트만**이 아니라 **객체**를 돌려줍니다.

| 필드 | 의미 |
|------|------|
| `response.choices[0].message.content` | **실제 답변 텍스트** ← 가장 많이 사용 |
| `response.choices[0].finish_reason` | `stop`=정상 종료, `length`=토큰 초과 |
| `response.usage.total_tokens` | 이번 호출에 쓴 토큰 수 (비용 참고) |
| `response.id` | 요청 ID (로그·디버깅용) |

In [ ]:
print('ID:', response.id)
print('finish_reason:', response.choices[0].finish_reason)
print('답변:', response.choices[0].message.content)
print('토큰 사용:', response.usage.total_tokens, '(prompt:', response.usage.prompt_tokens,
      '/ completion:', response.usage.completion_tokens, ')')

---
## 5. System / User 프롬프트 

같은 `user` 질문이라도 **`system`에 역할·출력 규칙**을 주면 답변 **형식·깊이·관점**이 달라집니다.

아래는 **의도적으로 짧고 모호한 질문**을 사용합니다. system 유무에 따라 차이가 잘 보입니다.

| | system | user |
|---|--------|------|
| 역할 | 모델의 **역할·규칙·형식** | 이번 **질문·작업** |
| 비유 | 직무 기술서 | 오늘 업무 지시 |

In [ ]:
# 질문을 짧고 모호하게 → system이 없으면 범용 답변, 있으면 역할·형식에 맞춘 답변
question = 'for문이 뭐야?'

r1 = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.2,
    messages=[{'role': 'user', 'content': question}],
)

print('=== system 없음 (범용 설명, 길어질 수 있음) ===')
print(r1.choices[0].message.content)

In [ ]:
r2 = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.2,
    messages=[
        {
            'role': 'system',
            'content': '''You are a Python tutor.
Rules:
- Answer in Korean only
- Use exactly 3 lines: [비유 1문장] / [코드 예시 max 3 lines] / [실무 한 줄]
- No long explanation, no bullet lists''',
        },
        {'role': 'user', 'content': question},
    ],
)

print('=== system 있음 (튜터 + 3줄 형식 강제) ===')
print(r2.choices[0].message.content)

---
## 6. temperature — **일반 예시**

같은 질문, 다른 `temperature` → 결과가 어떻게 달라지는지 확인합니다.

| temperature | 특징 | 용도 |
|-------------|------|------|
| 0 ~ 0.3 | 일관적, 사실 위주 | 분석, 판정, 코드 |
| 0.7 ~ 1.0 | 다양·창의적 | 브레인스토밍, 카피 |

In [30]:
question = '팀 워크숍 아이스브레이킹 아이디어 1가지만.'

for temp in [0.0, 0.7, 1.0]:
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=temp,
        messages=[{'role': 'user', 'content': question}],
    )
    print(f'[temperature={temp}] {r.choices[0].message.content}')
    print()

[temperature=0.0] "두 진실과 한 거짓" 게임을 제안합니다. 각 팀원이 자신에 대한 두 가지 진실과 하나의 거짓을 말합니다. 다른 팀원들은 어떤 것이 거짓인지 맞춰보는 방식입니다. 이 게임은 서로에 대한 이해를 높이고, 자연스럽게 대화를 유도하는 데 도움이 됩니다. 재미있고 가벼운 분위기를 만들어 팀원 간의 친밀감을 증진시킬 수 있습니다.

[temperature=0.7] "두 진실과 한 거짓" 게임을 추천합니다. 

각 팀원에게 자신에 대한 두 가지 진실과 하나의 거짓을 생각하게 합니다. 그런 다음 차례로 자신의 세 가지 이야기를 공유하고, 다른 팀원들은 어떤 것이 거짓인지 맞춰보는 방식입니다. 이 게임은 서로에 대한 이해를 높이고, 자연스럽게 대화를 유도하는 데 도움이 됩니다. 또한, 팀원들의 개성을 알 수 있는 재미있는 방법입니다!

[temperature=1.0] "두 진실과 한 거짓" 게임을 제안합니다. 

각 팀원이 차례로 자신에 대한 세 가지 사실을 공유합니다. 그 중 두 가지는 진짜이고, 하나는 거짓입니다. 다른 팀원들은 어떤 것이 거짓인지 맞추는 게임입니다. 이 활동은 서로에 대한 이해를 높이고, 대화를 촉진하는 데 도움이 됩니다. 분위기도 유쾌해져서 아이스브레이킹에 효과적입니다!



---
## 7. `max_tokens` — 출력 길이 제한

답변이 너무 길어지는 것을 막거나, 비용을 줄일 때 사용합니다.

In [ ]:
for max_tok in [20, 100]:
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=0.3,
        max_tokens=max_tok,
        messages=[{'role': 'user', 'content': '대한민국의 역사를 요약해줘.'}],
    )
    print(f'--- max_tokens={max_tok} (finish: {r.choices[0].finish_reason}) ---')
    print(r.choices[0].message.content)
    print()

---
## 8. `ask()` 함수 — 재사용 패턴

같은 API 호출을 함수로 감싸기.

In [ ]:
from typing import Any


def ask(user_message, system_message='You are a helpful assistant.', temperature=0.2, max_tokens=None):
    """1턴 질의 — 답변 텍스트만 반환"""
    kwargs = dict[str, str | float | list[dict[str, str] | dict[str, Any]]](
        model='gpt-4o-mini',
        temperature=temperature,
        messages=[
            {'role': 'system', 'content': system_message},
            {'role': 'user',   'content': user_message},
        ],
    )
    if max_tokens is not None:
        kwargs['max_tokens'] = max_tokens
    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message.content


print(ask('서울 3일 여행 코스 추천해줘. bullet 3개만.', max_tokens=150))

---
## 9. 멀티턴 대화 

이전 대화를 `messages`에 **그대로 이어 붙이면** 후속 질문이 가능합니다.

```
system → user → assistant → user → ...
```

### 기존대화

In [31]:
messages = [
    {'role': 'system', 'content': 'You are a helpful travel assistant. Answer in Korean.'},
    {'role': 'user',   'content': '제주도 2박 3일 여행 계획 짜줘. 하루 요약 1줄씩.'},
]

r1 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
answer1 = r1.choices[0].message.content
print('=== 1턴 ===')
print(answer1)


# 모델 응답 저장
messages.append({
    "role": "assistant",
    "content": answer1
})


messages.append(
    {'role': 'user',   'content': '2일차만 더 자세히. 맛집 2곳 포함.'},
)


r2 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
print('\n=== 2턴 (맥락 유지) ===')
print(r2.choices[0].message.content)

=== 1턴 ===
제주도 2박 3일 여행 계획은 다음과 같습니다:

**1일차:** 제주공항 도착 후 한라산 등반 및 성산일출봉에서 일몰 감상.

**2일차:** 우도 섬으로 배를 타고 가서 자전거 탐방 및 해변에서 여유로운 시간 보내기.

**3일차:** 제주 민속촌 방문 후 제주도 특산물 쇼핑 및 공항으로 귀환.

=== 2턴 (맥락 유지) ===
**2일차 상세 계획:**

- **아침:** 제주도에서 유명한 '미역국'으로 아침 식사 (추천 맛집: '제주미역국집').
  
- **오전:** 우도 섬으로 배를 타고 이동. 우도에 도착 후 자전거를 대여하여 섬을 탐방. 우도의 아름다운 해변과 자연 경관을 즐기세요.

- **점심:** 우도에서 유명한 '땅콩 아이스크림'과 '우도 땅콩국수'를 맛볼 수 있는 맛집 방문 (추천 맛집: '우도 땅콩국수').

- **오후:** 우도의 해변에서 수영이나 일광욕을 즐기고, 섬의 주요 명소인 '서빈백사'와 '우도봉'을 탐방.

- **저녁:** 제주도로 돌아와서 해산물 요리로 저녁 식사 (추천 맛집: '제주 해녀의 집').

- **밤:** 제주도의 아름다운 야경을 감상하며 하루를 마무리.


### 멀티턴

### Assistant 메시지는 앞서 있었던 prompt들(user/system)에 대한 모델의 응답을 구성하며, 종종 대화 상태를 유지하기 위해 히스토리의 일부로 저장됩니다.

#### 특징
모델이 생성한 응답을 포함한다.

이전 message의 context 처리를 위해 대화의 연속성 유지를 돕는다.

다음 API 호출에서는 assistant 메시지를 포함시켜야 문맥이 이어짐.

In [ ]:
messages = [
    {'role': 'system', 'content': 'You are a helpful travel assistant. Answer in Korean.'},
    {'role': 'user',   'content': '제주도 2박 3일 여행 계획 짜줘. 하루 요약 1줄씩.'},
]

r1 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
answer1 = r1.choices[0].message.content
print('=== 1턴 ===')
print(answer1)

messages.append({'role': 'assistant', 'content': answer1})
messages.append({'role': 'user', 'content': '2일차만 더 자세히. 맛집 2곳 포함.'})

r2 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
print('\n=== 2턴 (맥락 유지) ===')
print(r2.choices[0].message.content)

---
## ✏️ 실습 1

아래 주제로 **2턴** 대화를 만들어 보세요.

1턴: "Python for문 기본 문법 설명해줘"
2턴: "방금 설명한 걸로 1~5 합 구하는 코드 예시만 보여줘"

In [33]:
# ── 여기에 작성 ──
messages = [
    {'role': 'system', 'content': 'You are a Python tutor. Answer in Korean.'},
    {'role': 'user','content' : 'Python for문 기본 문법 설명해줘'}
]

r1=client.chat.completions.create(model='gpt-4o-mini',temperature=0.3,messages=messages)
answer1=r1.choices[0].message.content
messages.append({'role':'assistant','content':answer1})
messages.append({'role':'user','content':'방금 설명한걸로 1~5합 구하는 코드 예시만 보여줘'})
r2=client.chat.completions.create(model='gpt-4o-mini',temperature=0.3,messages=messages)
answer2=r2.choices[0].message.content
print(answer2)
# r1 = client.chat.completions.create(...)
# messages.append(...)
# r2 = client.chat.completions.create(...)

1부터 5까지의 합을 구하는 코드는 다음과 같습니다:

```python
total = 0
for i in range(1, 6):  # 1부터 5까지의 숫자를 생성
    total += i  # total에 i를 더함

print(total)  # 결과 출력
```

이 코드는 `total` 변수에 1부터 5까지의 숫자를 모두 더한 값을 저장하고, 마지막에 그 결과를 출력합니다. 결과는 15가 됩니다.


---
## ✏️ 실습 1

커서 프롬프트 : 3일차 폴더에 스트림릿과 실습코드의 openai api를 이용해서 간단한 챗봇을 만드는 python 코드 만들어줘

In [34]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 24.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.6/797.6 kB 13.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 14.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 17.3 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 15.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 14.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 15.8 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33/33 [streamlit]33 [streamlit]]malizer]
